# 04 - ML scores + Customer 360 (Gold)

Computes churn risk and cross-sell recommendations from the curated tables,
then builds the denormalized `customer_360` gold table used by the SQL endpoint
and the Fabric Data Agent. All tables are read from / written to the **`gold`** schema.

## Use the gold schema

Sets the current schema to `gold` so the statements below read the curated tables
and create the ML + `customer_360` tables in `gold`.

In [ ]:
spark.sql('CREATE SCHEMA IF NOT EXISTS gold')
spark.sql('USE gold')
print('current schema:', spark.catalog.currentDatabase())

## Churn model (trained in Fabric with MLflow)

Trains a scikit-learn classifier on `churn_label` (the observed-churn training
target), logs it to an **MLflow experiment**, **registers it as a Fabric ML model**,
then scores every account into `gold.ml_churn_score`. This replaces the old
rule-based SQL scoring with a real train -> register -> score pipeline.

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

FEATURES = ['tenure_months', 'autopay', 'credit_risk', 'product_count', 'total_mrc',
            'unpaid_ct', 'avg_uptime', 'avg_latency', 'avg_csat', 'cancel_ct']
feat = spark.sql('''SELECT
  a.account_id, a.customer_id,
  c.tenure_months,
  CAST(a.autopay AS INT) AS autopay,
  CASE a.credit_class WHEN 'A' THEN 0 WHEN 'B' THEN 1 ELSE 2 END AS credit_risk,
  COALESCE(s.product_count, 0) AS product_count,
  COALESCE(s.total_mrc, 0.0) AS total_mrc,
  COALESCE(u.unpaid_ct, 0) AS unpaid_ct,
  COALESCE(sm.avg_uptime, 100.0) AS avg_uptime,
  COALESCE(sm.avg_latency, 20.0) AS avg_latency,
  COALESCE(fb.avg_csat, 5.0) AS avg_csat,
  COALESCE(ct.cancel_ct, 0) AS cancel_ct,
  a.churn_label
FROM dim_account a
JOIN dim_customer c ON a.customer_id = c.customer_id
LEFT JOIN (SELECT account_id, COUNT(*) AS product_count, SUM(mrc) AS total_mrc
           FROM fact_subscription WHERE status = 'active' GROUP BY account_id) s ON a.account_id = s.account_id
LEFT JOIN (SELECT account_id, COUNT(*) AS unpaid_ct FROM fact_invoice WHERE paid = false
           GROUP BY account_id) u ON a.account_id = u.account_id
LEFT JOIN (SELECT account_id, AVG(uptime_pct) AS avg_uptime, AVG(latency_ms) AS avg_latency
           FROM fact_service_metric GROUP BY account_id) sm ON a.account_id = sm.account_id
LEFT JOIN (SELECT account_id, AVG(csat) AS avg_csat FROM fact_feedback GROUP BY account_id) fb
           ON a.account_id = fb.account_id
LEFT JOIN (SELECT customer_id, COUNT(*) AS cancel_ct FROM fact_contact WHERE reason = 'cancel_request'
           GROUP BY customer_id) ct ON a.customer_id = ct.customer_id''').toPandas()
X = feat[FEATURES].astype('float64')
y = feat['churn_label'].astype('int32')
print('accounts:', len(feat), '| churn rate:', round(float(y.mean()), 3))

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
model = GradientBoostingClassifier(random_state=42)
model.fit(X_tr, y_tr)
auc = float(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]))
print('Trained churn model | test AUC:', round(auc, 3))

In [ ]:
# Log + register to MLflow. Best-effort: can fail when the run identity lacks rights
# (e.g. a service principal in an automated job). Scoring below still works either way.
registered = False
try:
    mlflow.set_experiment('telco-churn')
    with mlflow.start_run(run_name='churn-gbc'):
        mlflow.log_param('model_type', 'GradientBoostingClassifier')
        mlflow.log_metric('test_auc', auc)
        mlflow.sklearn.log_model(model, artifact_path='model',
                                 registered_model_name='telco_churn_model')
    registered = True
    print('Logged run + registered telco_churn_model.')
except Exception as ex:
    print('MLflow logging/registration skipped (non-fatal):', ex)

In [ ]:
# Score every account. Use the registered model if available, else the in-memory model.
proba = None
if registered:
    try:
        from mlflow.tracking import MlflowClient
        _vs = MlflowClient().search_model_versions("name='telco_churn_model'")
        _v = max(_vs, key=lambda v: int(v.version)).version
        scorer = mlflow.sklearn.load_model(f'models:/telco_churn_model/{_v}')
        proba = scorer.predict_proba(X)[:, 1]
        print('Scored with registered model version', _v)
    except Exception as ex:
        print('registry load failed, using in-memory model:', ex)
if proba is None:
    proba = model.predict_proba(X)[:, 1]
    print('Scored with in-memory model.')
feat['churn_probability'] = np.round(proba, 3)
feat['risk_band'] = np.where(feat.churn_probability >= 0.55, 'High',
                     np.where(feat.churn_probability >= 0.30, 'Medium', 'Low'))

In [ ]:
# Per-account top reason: standardized feature x model importance -> biggest risk driver.
imp = dict(zip(FEATURES, model.feature_importances_))
Z = (X - X.mean()) / X.std(ddof=0).replace(0, 1.0)
risk_dir = {'tenure_months': -1, 'autopay': -1, 'credit_risk': 1, 'product_count': -1,
            'total_mrc': 0, 'unpaid_ct': 1, 'avg_uptime': -1, 'avg_latency': 1,
            'avg_csat': -1, 'cancel_ct': 1}
labels = {'tenure_months': 'short tenure', 'autopay': 'no autopay', 'credit_risk': 'credit risk',
          'unpaid_ct': 'unpaid invoices', 'avg_uptime': 'degraded service',
          'avg_latency': 'high latency', 'avg_csat': 'low satisfaction',
          'cancel_ct': 'cancellation intent', 'product_count': 'few products'}
contrib = pd.DataFrame({f: Z[f] * risk_dir[f] * imp[f] for f in labels}, index=Z.index)
feat['top_reason'] = contrib.idxmax(axis=1).map(labels)
feat.loc[feat.risk_band == 'Low', 'top_reason'] = 'stable account'

In [ ]:
from pyspark.sql import functions as F
# Coerce to native Python types (avoids Spark inferring from numpy dtypes in a job).
out_pdf = feat[['customer_id', 'account_id', 'churn_probability', 'risk_band', 'top_reason']].copy()
out_pdf['customer_id'] = out_pdf['customer_id'].astype(str)
out_pdf['account_id'] = out_pdf['account_id'].astype(str)
out_pdf['churn_probability'] = out_pdf['churn_probability'].astype(float)
out_pdf['risk_band'] = out_pdf['risk_band'].astype(str)
out_pdf['top_reason'] = out_pdf['top_reason'].astype(str)
score_df = spark.createDataFrame(out_pdf).withColumn('scored_date', F.current_date())
(score_df.write.format('delta').mode('overwrite')
    .option('overwriteSchema', 'true').saveAsTable('gold.ml_churn_score'))
print('gold.ml_churn_score rows:', score_df.count())
spark.sql('SELECT risk_band, COUNT(*) c FROM gold.ml_churn_score GROUP BY risk_band ORDER BY 1').show()

## Cross-sell recommendations (product-gap logic)

In [ ]:
spark.sql('''CREATE OR REPLACE TABLE ml_crosssell_reco AS
WITH owned AS (
    SELECT account_id, collect_set(product_id) AS products
    FROM fact_subscription GROUP BY account_id
)
SELECT
    a.account_id,
    CASE WHEN NOT array_contains(o.products, 'PROD_MOB') THEN 'PROD_MOB'
         WHEN NOT array_contains(o.products, 'PROD_TV')  THEN 'PROD_TV'
         ELSE 'PROD_VOICE' END AS recommended_product_id,
    CASE WHEN NOT array_contains(o.products, 'PROD_MOB') THEN 'PROMO_XSELL_1'
         WHEN NOT array_contains(o.products, 'PROD_TV')  THEN 'PROMO_XSELL_2'
         ELSE NULL END AS recommended_promotion_id,
    ROUND(0.4 + LEAST(c.tenure_months, 60) / 200.0, 3) AS score,
    CASE WHEN NOT array_contains(o.products, 'PROD_MOB')
              THEN 'Internet customer without mobile'
         WHEN NOT array_contains(o.products, 'PROD_TV')
              THEN 'Eligible for TV + internet bundle'
         ELSE 'Add home phone to bundle' END AS rationale,
    CURRENT_DATE() AS scored_date
FROM dim_account a
JOIN dim_customer c ON a.customer_id = c.customer_id
JOIN owned o ON a.account_id = o.account_id
WHERE a.status <> 'cancelled'
  AND NOT (array_contains(o.products, 'PROD_MOB')
           AND array_contains(o.products, 'PROD_TV')
           AND array_contains(o.products, 'PROD_VOICE'))''')
print('ml_crosssell_reco rows:', spark.table('ml_crosssell_reco').count())
spark.sql('SELECT recommended_product_id, COUNT(*) c FROM ml_crosssell_reco GROUP BY recommended_product_id').show()

## Customer 360 (gold serving object)

In [ ]:
spark.sql('''CREATE OR REPLACE TABLE customer_360 AS
WITH sub_agg AS (
    SELECT account_id,
           COUNT(*) AS active_products,
           ROUND(SUM(mrc), 2) AS total_mrc,
           concat_ws(', ', collect_list(product_id)) AS product_list
    FROM fact_subscription WHERE status = 'active' GROUP BY account_id
),
latest_inv AS (
    SELECT account_id, period_end, amount, due_date, paid, is_first_bill
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY period_end DESC) rn
        FROM fact_invoice
    ) WHERE rn = 1
),
balance AS (
    SELECT account_id, ROUND(SUM(amount), 2) AS open_balance
    FROM fact_invoice WHERE paid = false GROUP BY account_id
),
open_wo AS (
    SELECT account_id, COUNT(*) AS open_work_orders
    FROM fact_work_order WHERE status = 'open' GROUP BY account_id
),
last_contact AS (
    SELECT customer_id, MAX(contact_ts) AS last_contact_ts
    FROM fact_contact GROUP BY customer_id
),
recent_outage AS (
    SELECT DISTINCT geo_id FROM fact_outage
    WHERE start_time >= date_sub(current_timestamp(), 14)
)
SELECT
    c.customer_id, a.account_id,
    c.first_name, c.last_name, c.email, c.phone, c.segment,
    c.tenure_months, a.status AS account_status, a.open_date, a.is_new_customer,
    a.autopay, a.credit_class,
    g.city, g.state, g.region, g.zip,
    COALESCE(s.active_products, 0) AS active_products,
    COALESCE(s.total_mrc, 0) AS total_mrc,
    s.product_list,
    li.amount AS last_invoice_amount, li.due_date AS last_invoice_due,
    li.paid AS last_invoice_paid, li.is_first_bill AS last_invoice_is_first_bill,
    COALESCE(b.open_balance, 0) AS open_balance,
    COALESCE(w.open_work_orders, 0) AS open_work_orders,
    lc.last_contact_ts,
    CASE WHEN ro.geo_id IS NOT NULL THEN true ELSE false END AS recent_outage_exposure,
    ch.churn_probability, ch.risk_band, ch.top_reason AS churn_top_reason,
    xs.recommended_product_id AS top_crosssell_product,
    xs.recommended_promotion_id AS top_crosssell_promo,
    xs.score AS top_crosssell_score
FROM dim_customer c
JOIN dim_account a ON c.customer_id = a.customer_id
LEFT JOIN dim_geography g ON c.geo_id = g.geo_id
LEFT JOIN sub_agg s ON a.account_id = s.account_id
LEFT JOIN latest_inv li ON a.account_id = li.account_id
LEFT JOIN balance b ON a.account_id = b.account_id
LEFT JOIN open_wo w ON a.account_id = w.account_id
LEFT JOIN last_contact lc ON c.customer_id = lc.customer_id
LEFT JOIN recent_outage ro ON c.geo_id = ro.geo_id
LEFT JOIN ml_churn_score ch ON c.customer_id = ch.customer_id
LEFT JOIN ml_crosssell_reco xs ON a.account_id = xs.account_id''')
print('customer_360 rows:', spark.table('customer_360').count())

## Validation: sample a new customer with an unpaid first bill

In [ ]:
spark.sql('''
SELECT customer_id, first_name, last_name, account_status, tenure_months,
       last_invoice_amount, last_invoice_is_first_bill, open_balance,
       risk_band, top_crosssell_product
FROM customer_360
WHERE last_invoice_is_first_bill = true AND last_invoice_paid = false
LIMIT 10''').show(truncate=False)